# 12. RAG Ablation Lab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/12_rag_ablation_lab.ipynb)

This notebook turns the ideas from the earlier RAG track into a compact local benchmark. You will vary chunking, embedding text, reranking, and context budget, then measure how those choices change end-to-end behavior.

## Learning goals

- build multiple retrieval stores from the same corpus
- compare chunking strategies and embedding text variants
- add a simple reranker and explicit context budget
- run a tiny end-to-end RAG benchmark and summarize the results

## Experimental mindset

An ablation answers one question at a time: what changed, and did that change help? We will keep the corpus and benchmark fixed while changing one or two retrieval knobs at a time.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import faiss
import numpy as np
import torch
from google.colab import drive
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def embed_texts(texts, batch_size=32):
    if isinstance(texts, str):
        texts = [texts]
    return embed_model.encode(
        list(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

def build_faiss_index(texts):
    texts = list(texts)
    embeddings = embed_texts(texts).astype(np.float32)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return {
        "texts": texts,
        "embeddings": embeddings,
        "index": index,
    }

def search_index(query, store, k=5):
    k = min(k, len(store["texts"]))
    if k <= 0:
        return []

    query_vector = embed_texts([query]).astype(np.float32)
    scores, indices = store["index"].search(query_vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append(
            {
                "index": int(idx),
                "score": float(score),
                "text": store["texts"][idx],
            }
        )
    return results

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"✓ Embedding model loaded: {EMBEDDING_MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 1 — Shared helpers for the lab

The benchmark stays intentionally small so the full experiment is runnable in Colab. The helpers below keep the tables readable and the scoring transparent.

In [ ]:
from collections import defaultdict
import re

random.seed(17)
GEN_CACHE = {}

STOPWORDS = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "with", "by",
    "at", "from", "during", "into", "over", "under", "after", "before", "than",
    "what", "which", "where", "when", "how", "did", "does", "is", "are", "was",
    "were", "be", "been", "it", "its", "their", "that", "this", "these", "those"
}


def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def content_words(text):
    return [token for token in normalize(text).split() if token not in STOPWORDS]


def split_sentences(text):
    cleaned = text.replace("\n", " ").strip()
    parts = re.split(r"(?<=[.!?])\s+", cleaned)
    return [part.strip() for part in parts if part.strip()]


def extract_numbers(text):
    return set(re.findall(r"\d+(?:\.\d+)?", text))


def clip(text, width=82):
    text = str(text)
    return text if len(text) <= width else text[: width - 3] + "..."


def print_rows(rows, columns=None, limit=None):
    rows = list(rows)
    if limit is not None:
        rows = rows[:limit]
    if not rows:
        print("(no rows)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {}
    for column in columns:
        widths[column] = len(str(column))
        for row in rows:
            widths[column] = max(widths[column], len(str(row.get(column, ""))))
    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

## Step 2 — Local corpus for the ablation study

The corpus reuses the same infrastructure documents from earlier notebooks so the experiments connect naturally to the broader RAG course.

In [ ]:
CORPUS = [
    {
        "doc_id": "D1",
        "title": "Harbor District Microgrid Pilot",
        "text": "The Harbor district launched a resilience pilot in 2024. It installed 6 megawatts of rooftop solar and a 20 megawatt-hour battery. During two grid outages, the microgrid kept the health clinic and water pumps online for 11 hours. Operators reported a 34 percent reduction in diesel generator use compared with the previous summer.",
    },
    {
        "doc_id": "D2",
        "title": "Cedar Wastewater Heat Recovery Memo",
        "text": "The Cedar treatment plant now captures heat from outgoing effluent. The recovered heat keeps digesters warm and supplies a nearby greenhouse loop. Plant managers reported an 18 percent reduction in natural gas consumption. The memo estimates a payback period of about 5.5 years.",
    },
    {
        "doc_id": "D3",
        "title": "Northwind Offshore Wind Siting Brief",
        "text": "The Northwind team shifted turbine placement 9 kilometers east to avoid a major bird migration corridor. The export cable shares an existing shipping channel for the first 14 kilometers. Environmental review says construction noise is the largest short-term risk. The project brief estimates annual output could power 210000 homes.",
    },
    {
        "doc_id": "D4",
        "title": "Library Retrofit Case Study",
        "text": "A central library replaced chillers with ground-source heat pumps and smart ventilation controls. Energy use intensity fell from 212 to 148 kilowatt-hours per square meter. Summer comfort complaints dropped by 40 percent. Total project cost was 4.2 million dollars with an 11-year payback.",
    },
    {
        "doc_id": "D5",
        "title": "Drought Response Handbook",
        "text": "Stage two restrictions begin when reservoir storage stays below 62 percent for 21 consecutive days. The handbook says leak repairs can save up to 12 percent of summer demand. Recharge basins should rest every seventh day to prevent compaction. Public messaging focuses on irrigation schedules and the repair hotline.",
    },
    {
        "doc_id": "D6",
        "title": "Transit Electrification Plan",
        "text": "Phase one purchases 18 battery buses by 2026. The full program targets 42 electric buses by 2028. Overnight depot charging serves most vehicles, while an on-route pantograph at Central Station handles the peak corridor. The plan forecasts 22 percent lower maintenance cost than diesel operations.",
    },
    {
        "doc_id": "D7",
        "title": "Estuary Flood Barrier Memo",
        "text": "The estuary barrier closes only during storm-surge warnings above 1.7 meters. Wetlands restoration is expected to delay peak flooding by 35 minutes. The annual maintenance budget is 8.4 million dollars. Fishing access is preserved through a side channel.",
    },
    {
        "doc_id": "D8",
        "title": "Data Center Water Reuse Project",
        "text": "A reclaim system now reuses 68 percent of cooling water at the municipal data center. The facility pairs with treated greywater from the wastewater plant. Summer potable water demand fell by 41 percent after deployment. Filtration membranes are replaced every 18 months.",
    },
]

print_rows(
    [
        {
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "words": len(doc["text"].split()),
        }
        for doc in CORPUS
    ],
    columns=["doc_id", "title", "words"],
)

In [ ]:
BENCHMARK = [
    {
        "qid": "B1",
        "question": "How much storage did the Harbor microgrid include and how long did it keep critical services online?",
        "required_docs": ["D1"],
        "facts": [
            [["20", "mwh"], ["20", "megawatt", "hour"]],
            [["11", "hours"], ["11", "hour"]],
        ],
    },
    {
        "qid": "B2",
        "question": "When do stage two drought restrictions begin and how often should recharge basins rest?",
        "required_docs": ["D5"],
        "facts": [
            [["62", "21", "days"], ["62", "percent", "21", "days"]],
            [["seventh", "day"], ["7th", "day"]],
        ],
    },
    {
        "qid": "B3",
        "question": "What siting change did Northwind make and how many homes could the project power annually?",
        "required_docs": ["D3"],
        "facts": [
            [["9", "east"], ["9", "kilometers", "east"]],
            [["210000", "homes"], ["210", "000", "homes"]],
        ],
    },
    {
        "qid": "B4",
        "question": "How many electric buses are targeted by 2028 and what charging approach handles peak service?",
        "required_docs": ["D6"],
        "facts": [
            [["42", "2028"], ["42", "electric", "buses", "2028"]],
            [["pantograph", "central", "station"]],
        ],
    },
    {
        "qid": "B5",
        "question": "Which projects reduced natural gas use or potable water demand through reuse, and by what percentages?",
        "required_docs": ["D2", "D8"],
        "facts": [
            [["18", "natural", "gas"], ["18", "percent", "natural", "gas"]],
            [["41", "potable", "water"], ["41", "percent", "potable", "water"]],
        ],
    },
]

print_rows(
    [
        {
            "qid": item["qid"],
            "question": clip(item["question"], 76),
            "required_docs": ", ".join(item["required_docs"]),
        }
        for item in BENCHMARK
    ],
    columns=["qid", "question", "required_docs"],
)

## Step 3 — Build chunking variants

We will compare full-document retrieval with smaller sliding-window chunks. Smaller chunks often improve precision, but they also create more candidates and can increase context assembly complexity.

In [ ]:
def sliding_windows(sentences, size, overlap):
    windows = []
    step = max(1, size - overlap)
    start = 0
    while start < len(sentences):
        window = sentences[start : start + size]
        if not window:
            break
        windows.append(window)
        if start + size >= len(sentences):
            break
        start += step
    return windows


def make_chunks(corpus, mode):
    chunks = []
    for doc in corpus:
        sentences = split_sentences(doc["text"])
        if mode == "full_doc":
            windows = [sentences]
        elif mode == "window2":
            windows = sliding_windows(sentences, size=2, overlap=1)
        elif mode == "window3":
            windows = sliding_windows(sentences, size=3, overlap=1)
        else:
            raise ValueError("Unknown mode: {}".format(mode))

        for idx, window in enumerate(windows, start=1):
            chunks.append(
                {
                    "chunk_id": "{}-C{}".format(doc["doc_id"], idx),
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "text": " ".join(window),
                }
            )
    return chunks


def embedding_text(chunk, style):
    if style == "body_only":
        return chunk["text"]
    if style == "title_prefixed":
        return chunk["title"] + ". " + chunk["text"]
    raise ValueError("Unknown embedding style: {}".format(style))

In [ ]:
def build_retriever(corpus, chunk_mode, embedding_style):
    chunks = make_chunks(corpus, chunk_mode)
    texts = [embedding_text(chunk, embedding_style) for chunk in chunks]
    return {
        "chunk_mode": chunk_mode,
        "embedding_style": embedding_style,
        "chunks": chunks,
        "store": build_faiss_index(texts),
    }


RETRIEVERS = {
    ("full_doc", "body_only"): build_retriever(CORPUS, "full_doc", "body_only"),
    ("window2", "body_only"): build_retriever(CORPUS, "window2", "body_only"),
    ("window3", "body_only"): build_retriever(CORPUS, "window3", "body_only"),
    ("window2", "title_prefixed"): build_retriever(CORPUS, "window2", "title_prefixed"),
}

chunk_rows = []
for (chunk_mode, embedding_style), retriever in RETRIEVERS.items():
    chunk_rows.append(
        {
            "chunk_mode": chunk_mode,
            "embedding_style": embedding_style,
            "chunk_count": len(retriever["chunks"]),
            "avg_words": round(statistics.mean(len(chunk["text"].split()) for chunk in retriever["chunks"]), 1),
        }
    )

print_rows(chunk_rows, columns=["chunk_mode", "embedding_style", "chunk_count", "avg_words"])

## Step 4 — Add a simple reranker

Dense retrieval gives a coarse shortlist. We can then rerank that shortlist with cheap lexical signals so exact query words and numbers matter a bit more.

In [ ]:
def lexical_overlap_score(query, text):
    query_terms = set(content_words(query))
    text_terms = set(content_words(text))
    if not query_terms:
        return 0.0
    return len(query_terms & text_terms) / len(query_terms)


def rerank_candidates(query, candidates):
    reranked = []
    for candidate in candidates:
        lexical = lexical_overlap_score(query, candidate["title"] + " " + candidate["text"])
        query_numbers = extract_numbers(query)
        if query_numbers:
            numeric = len(query_numbers & extract_numbers(candidate["text"])) / len(query_numbers)
        else:
            numeric = 0.0
        rerank_score = candidate["dense_score"] + 0.35 * lexical + 0.15 * numeric
        updated = dict(candidate)
        updated["lexical_score"] = round(lexical, 3)
        updated["rerank_score"] = round(rerank_score, 3)
        reranked.append(updated)
    return sorted(reranked, key=lambda row: row["rerank_score"], reverse=True)

## Step 5 — Add context budgeting and answer generation

A retriever can find many useful chunks, but the generator only sees what fits inside the prompt. Context budgeting turns retrieval quality into a practical systems decision.

In [ ]:
def dense_retrieve(query, retriever, k=6):
    results = search_index(query, retriever["store"], k=min(k, len(retriever["chunks"])))
    rows = []
    for result in results:
        chunk = retriever["chunks"][result["index"]]
        rows.append(
            {
                "chunk_id": chunk["chunk_id"],
                "doc_id": chunk["doc_id"],
                "title": chunk["title"],
                "text": chunk["text"],
                "dense_score": round(result["score"], 4),
            }
        )
    return rows


def apply_budget(candidates, word_budget):
    selected = []
    used = 0
    for candidate in candidates:
        words = len(candidate["text"].split())
        if selected and used + words > word_budget:
            continue
        if not selected and words > word_budget:
            selected.append(candidate)
            break
        if used + words <= word_budget:
            selected.append(candidate)
            used += words
    return selected


def build_context(chunks):
    blocks = []
    for chunk in chunks:
        blocks.append("[{0}] {1} ({2})".format(chunk["chunk_id"], chunk["title"], chunk["doc_id"]))
        blocks.append(chunk["text"])
        blocks.append("")
    return "\n".join(blocks).strip()


def answer_with_rag(question, selected_chunks, cache_key):
    prompt = """Answer the question using only the context below.
If the context is incomplete, say what is missing.
Cite supporting documents with tags like [D1].
Question: {question}

Context:
{context}

Answer in 2 to 4 sentences.""".format(
        question=question,
        context=build_context(selected_chunks),
    )
    if cache_key not in GEN_CACHE:
        GEN_CACHE[cache_key] = generate(prompt, max_new_tokens=160, temperature=0.0, do_sample=False, top_k=1)
    return GEN_CACHE[cache_key]

## Step 6 — Score answers from scratch

We will combine four lightweight measurements:

- retrieval_recall: were the needed documents retrieved?
- fact_recall: did the answer mention the key facts?
- citation_recall: did the answer cite the needed docs?
- context_support: do the answer claims overlap with the chosen context?

In [ ]:
def answer_claims(answer):
    claims = []
    for sentence in split_sentences(answer):
        cleaned = re.sub(r"\[[^\]]+\]", "", sentence).strip()
        if cleaned:
            claims.append(cleaned)
    return claims


def context_support_ratio(answer, selected_chunks):
    claims = answer_claims(answer)
    if not claims:
        return 0.0
    scores = []
    for claim in claims:
        best = 0.0
        for chunk in selected_chunks:
            lexical = lexical_overlap_score(claim, chunk["text"])
            claim_numbers = extract_numbers(claim)
            if claim_numbers:
                numeric = len(claim_numbers & extract_numbers(chunk["text"])) / len(claim_numbers)
            else:
                numeric = 0.0
            best = max(best, 0.75 * lexical + 0.25 * numeric)
        scores.append(best)
    return round(statistics.mean(scores), 3)


def extract_doc_citations(answer):
    matches = re.findall(r"\[([A-Z0-9,\s]+)\]", answer)
    doc_ids = []
    for group in matches:
        for part in group.split(","):
            part = part.strip()
            if part:
                doc_ids.append(part)
    return sorted(set(doc_ids))


def fact_recall(answer, fact_groups):
    answer_tokens = set(content_words(answer))
    hits = 0
    for alternatives in fact_groups:
        matched = False
        for tokens in alternatives:
            if set(tokens).issubset(answer_tokens):
                matched = True
                break
        if matched:
            hits += 1
    return hits / len(fact_groups)


def score_run(item, selected_chunks, answer):
    selected_docs = set(chunk["doc_id"] for chunk in selected_chunks)
    required_docs = set(item["required_docs"])
    cited_docs = set(extract_doc_citations(answer))
    retrieval_recall = len(required_docs & selected_docs) / len(required_docs)
    citation_recall = len(required_docs & cited_docs) / len(required_docs)
    fact_score = fact_recall(answer, item["facts"])
    support_score = context_support_ratio(answer, selected_chunks)
    overall = round(0.35 * retrieval_recall + 0.35 * fact_score + 0.15 * citation_recall + 0.15 * support_score, 3)
    return {
        "retrieval_recall": round(retrieval_recall, 3),
        "fact_recall": round(fact_score, 3),
        "citation_recall": round(citation_recall, 3),
        "context_support": round(support_score, 3),
        "overall": overall,
    }

## Step 7 — Run one demo variant end to end

Before launching the full grid, inspect a single pipeline run so you can see the selected chunks, the answer, and the resulting scores.

In [ ]:
demo_variant = {
    "name": "window2_title_rerank_180",
    "chunk_mode": "window2",
    "embedding_style": "title_prefixed",
    "rerank": True,
    "word_budget": 180,
    "initial_k": 6,
}

demo_item = BENCHMARK[0]
demo_retriever = RETRIEVERS[(demo_variant["chunk_mode"], demo_variant["embedding_style"])]
demo_candidates = dense_retrieve(demo_item["question"], demo_retriever, k=demo_variant["initial_k"])
demo_ranked = rerank_candidates(demo_item["question"], demo_candidates) if demo_variant["rerank"] else demo_candidates
demo_selected = apply_budget(demo_ranked, demo_variant["word_budget"])
demo_answer = answer_with_rag(demo_item["question"], demo_selected, cache_key=demo_variant["name"] + "::" + demo_item["qid"])

print_rows(demo_selected, columns=["chunk_id", "doc_id", "dense_score", "text"])
print("\nGenerated answer:\n")
print(demo_answer)
print("\nScores:", score_run(demo_item, demo_selected, demo_answer))

## Step 8 — Define the ablation grid

Each variant changes one or more levers. Notice how the grid covers chunking, embedding text, reranking, and budget without exploding into a huge experiment.

In [ ]:
VARIANTS = [
    {
        "name": "full_doc_body_180",
        "chunk_mode": "full_doc",
        "embedding_style": "body_only",
        "rerank": False,
        "word_budget": 180,
        "initial_k": 4,
    },
    {
        "name": "window2_body_180",
        "chunk_mode": "window2",
        "embedding_style": "body_only",
        "rerank": False,
        "word_budget": 180,
        "initial_k": 6,
    },
    {
        "name": "window3_body_180",
        "chunk_mode": "window3",
        "embedding_style": "body_only",
        "rerank": False,
        "word_budget": 180,
        "initial_k": 6,
    },
    {
        "name": "window2_title_180",
        "chunk_mode": "window2",
        "embedding_style": "title_prefixed",
        "rerank": False,
        "word_budget": 180,
        "initial_k": 6,
    },
    {
        "name": "window2_title_rerank_180",
        "chunk_mode": "window2",
        "embedding_style": "title_prefixed",
        "rerank": True,
        "word_budget": 180,
        "initial_k": 6,
    },
    {
        "name": "window2_title_rerank_90",
        "chunk_mode": "window2",
        "embedding_style": "title_prefixed",
        "rerank": True,
        "word_budget": 90,
        "initial_k": 6,
    },
]

print_rows(VARIANTS, columns=["name", "chunk_mode", "embedding_style", "rerank", "word_budget", "initial_k"])

## Step 9 — Run the benchmark

This is the expensive cell in the notebook because it performs generation for every variant-question pair. The cache keeps reruns manageable.

In [ ]:
all_rows = []
for variant in VARIANTS:
    retriever = RETRIEVERS[(variant["chunk_mode"], variant["embedding_style"])]
    for item in BENCHMARK:
        candidates = dense_retrieve(item["question"], retriever, k=variant["initial_k"])
        ranked = rerank_candidates(item["question"], candidates) if variant["rerank"] else candidates
        selected = apply_budget(ranked, variant["word_budget"])
        answer = answer_with_rag(item["question"], selected, cache_key=variant["name"] + "::" + item["qid"])
        scored = score_run(item, selected, answer)
        all_rows.append(
            {
                "variant": variant["name"],
                "qid": item["qid"],
                "selected_docs": ", ".join(sorted(set(chunk["doc_id"] for chunk in selected))),
                "retrieval_recall": scored["retrieval_recall"],
                "fact_recall": scored["fact_recall"],
                "citation_recall": scored["citation_recall"],
                "context_support": scored["context_support"],
                "overall": scored["overall"],
                "answer": clip(answer, 92),
            }
        )

print_rows(all_rows[:12], columns=["variant", "qid", "selected_docs", "retrieval_recall", "fact_recall", "citation_recall", "context_support", "overall", "answer"])

## Step 10 — Summarize variant performance

The aggregate table below is the ablation result you would keep in an experiment log. It is small, local, and easy to rerun after prompt or retrieval changes.

In [ ]:
def summarize_runs(rows):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["variant"]].append(row)
    summary = []
    for variant, items in grouped.items():
        summary.append(
            {
                "variant": variant,
                "retrieval_recall": round(statistics.mean(row["retrieval_recall"] for row in items), 3),
                "fact_recall": round(statistics.mean(row["fact_recall"] for row in items), 3),
                "citation_recall": round(statistics.mean(row["citation_recall"] for row in items), 3),
                "context_support": round(statistics.mean(row["context_support"] for row in items), 3),
                "overall": round(statistics.mean(row["overall"] for row in items), 3),
            }
        )
    return sorted(summary, key=lambda row: row["overall"], reverse=True)


summary_rows = summarize_runs(all_rows)
print_rows(summary_rows, columns=["variant", "retrieval_recall", "fact_recall", "citation_recall", "context_support", "overall"])

## Step 11 — Inspect the weakest runs

When a variant underperforms, inspect the worst question-level rows instead of immediately tuning everything at once.

In [ ]:
weak_rows = sorted(all_rows, key=lambda row: row["overall"])[:8]
print_rows(weak_rows, columns=["variant", "qid", "selected_docs", "retrieval_recall", "fact_recall", "citation_recall", "context_support", "overall", "answer"])

## Step 12 — Compare context budgets directly

The two reranked title-aware variants differ only in budget. That makes them a clean mini-study of context compression versus answer quality.

In [ ]:
budget_rows = [
    row for row in all_rows
    if row["variant"] in {"window2_title_rerank_180", "window2_title_rerank_90"}
]
print_rows(budget_rows, columns=["variant", "qid", "selected_docs", "retrieval_recall", "fact_recall", "citation_recall", "context_support", "overall"])

## Step 13 — Short interpretation checklist

When you read the tables, ask four questions:

- did smaller chunks improve retrieval recall or just add noise?
- did title-prefixed embeddings help route the query to the right document family?
- did reranking improve precision enough to matter downstream?
- how much answer quality did you lose when the context budget got tighter?

In [ ]:
best_variant = summarize_runs(all_rows)[0]["variant"]
print("Best variant:", best_variant)
print()
for row in all_rows:
    if row["variant"] == best_variant:
        print(row["qid"], "->", row["answer"])
        print()

## Recap

You now have a compact but real RAG ablation lab: multiple retrieval stores, chunking variants, embedding text variants, reranking, budgeting, and end-to-end benchmark tables. This is the core workflow you can reuse when you iterate on larger RAG systems later.